# Manufacturing Multi-Agent — Three Sub-Agents

Builds and tests the 3 specialized sub-agents:
1. **Defect Classifier** — ai_query on inspection notes → severity + root cause
2. **Service Bulletin Retriever** — Vector Search RAG → relevant fixes
3. **Quality SQL Agent** — Natural language → SQL → structured data answers

**Databricks AI Features:** Foundation Models, ai_query, Vector Search, MLflow, Text-to-SQL


Install LangChain packages first. `%pip` restarts Python, so it must run before `_config` sets up widgets and `sys.path`.


In [ ]:
%pip install databricks-langchain langchain-core -q


## Setup


In [ ]:
%run ./_config


Re-read job widgets in this notebook so catalog and schema are defined after `%run`.


In [ ]:
# Job parameters / widgets must be read in this notebook. %run does not always export names from .ipynb.
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
try:
    llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
except Exception:
    llm_endpoint = "databricks-claude-sonnet-4-5"
try:
    embedding_endpoint = dbutils.widgets.get("embedding_endpoint").strip()
except Exception:
    embedding_endpoint = "databricks-gte-large-en"
try:
    vs_endpoint = dbutils.widgets.get("vs_endpoint").strip()
except Exception:
    vs_endpoint = "mfg_quality_vs_endpoint"
try:
    eval_sample_size = dbutils.widgets.get("eval_sample_size").strip() or "50"
except Exception:
    eval_sample_size = "50"
try:
    cost_owner = dbutils.widgets.get("cost_owner").strip()
except Exception:
    cost_owner = ""
VS_INDEX_NAME = f"{catalog}.{schema}.service_bulletins_index"
print(f"Bound widgets: {catalog}.{schema}")


SQL guard (`sanitize_sql`) is loaded by `_config` from `src/sql_guard.py`.


Set the notebook session to the demo schema.


In [ ]:
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
LLM_MODEL = llm_endpoint
VS_ENDPOINT_NAME = vs_endpoint


## Sub-Agent 1: Defect Classifier (ai_query)

Classifies inspection notes into severity + root cause using constrained prompting.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW classified_inspections AS
SELECT
  inspection_id,
  part_type,
  plant_name,
  line,
  defect_notes,
  expected_severity,
  expected_root_cause,
  ai_query(
    '{LLM_MODEL}',
    CONCAT(
      'Classify this manufacturing inspection using ONLY these categories:

SEVERITY (pick exactly one):
  Critical = safety risk, batch quarantine needed, line shutdown
  Major = out of spec, rework required before shipment
  Minor = borderline or cosmetic, log and monitor
  None = within spec, approved

ROOT CAUSE (pick exactly one, or None if severity is None):
  Material = raw material, supplier, contamination
  Process = temperature, pressure, feed rate, cycle time, process drift
  Equipment = tool wear, calibration, vibration, machine malfunction
  Human = operator error, missed step, wrong setup

Return ONLY a JSON object:
{{"severity": "...", "root_cause": "...", "corrective_action": "one sentence"}}

INSPECTION DATA:
', defect_notes)
  ) AS ai_output
FROM {catalog}.{schema}.quality_inspections
LIMIT 50
""")


Parse classifier JSON (or keyword fallback) and score against labels.


In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW classifier_eval AS
WITH parsed AS (
  SELECT *,
    CASE
      WHEN ai_output RLIKE '"severity"\\s*:\\s*"(Critical|Major|Minor|None)"'
        THEN regexp_extract(ai_output, '"severity"\\s*:\\s*"(Critical|Major|Minor|None)"', 1)
      WHEN lower(ai_output) RLIKE '(critical|safety|quarantine|shutdown)'   THEN 'Critical'
      WHEN lower(ai_output) RLIKE '(major|reject|rework|out of spec)'       THEN 'Major'
      WHEN lower(ai_output) RLIKE '(minor|borderline|cosmetic|monitor)'     THEN 'Minor'
      WHEN lower(ai_output) RLIKE '(none|pass|approved|within spec)'        THEN 'None'
      ELSE 'UNPARSEABLE'
    END AS parsed_severity,
    CASE
      WHEN ai_output RLIKE '"root_cause"\\s*:\\s*"(Material|Process|Equipment|Human|None)"'
        THEN regexp_extract(ai_output, '"root_cause"\\s*:\\s*"(Material|Process|Equipment|Human|None)"', 1)
      WHEN lower(ai_output) RLIKE '(material|supplier|contamina)'   THEN 'Material'
      WHEN lower(ai_output) RLIKE '(process|temperature|pressure|feed rate)' THEN 'Process'
      WHEN lower(ai_output) RLIKE '(equipment|tool|calibrat|machine)'  THEN 'Equipment'
      WHEN lower(ai_output) RLIKE '(human|operator|missed|wrong setup)' THEN 'Human'
      ELSE 'None'
    END AS parsed_root_cause
  FROM classified_inspections
)
SELECT *,
  (parsed_severity = expected_severity) AS severity_correct,
  (parsed_root_cause = expected_root_cause) AS root_cause_correct,
  (parsed_severity = expected_severity AND parsed_root_cause = expected_root_cause) AS both_correct
FROM parsed


Classifier scorecard.


In [ ]:
%sql
SELECT
  COUNT(*) AS total,
  ROUND(AVG(CAST(severity_correct AS INT)) * 100, 1) AS severity_accuracy_pct,
  ROUND(AVG(CAST(root_cause_correct AS INT)) * 100, 1) AS root_cause_accuracy_pct,
  ROUND(AVG(CAST(both_correct AS INT)) * 100, 1) AS both_correct_pct,
  SUM(CASE WHEN parsed_severity = 'UNPARSEABLE' THEN 1 ELSE 0 END) AS unparseable_count
FROM classifier_eval


## Sub-Agent 2: Service Bulletin Retriever (Vector Search RAG)

Given a defect description, retrieves relevant service bulletins and synthesizes an answer.


In [ ]:
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import mlflow

# Vector Search retriever
retriever = DatabricksVectorSearch(
    endpoint=VS_ENDPOINT_NAME,
    index_name=VS_INDEX_NAME,
    columns=["bulletin_id", "part_type", "failure_mode", "severity_level", "priority"],
).as_retriever(search_kwargs={"k": 3})

# Foundation Model for answer synthesis
llm = ChatDatabricks(endpoint=LLM_MODEL, temperature=0.1)

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""You are a manufacturing quality assistant.
A technician has reported a defect. Based on the relevant service bulletins below, provide:
1. The most likely root cause
2. Recommended corrective action (step by step)
3. Which service bulletin(s) to reference

Be specific and cite bulletin IDs.

REPORTED DEFECT:
{question}

RELEVANT SERVICE BULLETINS:
{context}

Respond in JSON:
{{"root_cause": "...", "corrective_action": "...", "referenced_bulletins": ["SB-..."], "confidence": "high/medium/low"}}
""")

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{doc.metadata['bulletin_id']}] {doc.metadata['part_type']} - {doc.metadata['failure_mode']}\n{doc.page_content}"
        for doc in docs
    )

# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)


Run sample retrieval questions against the RAG chain.


In [ ]:
# Test the retriever
test_queries = [
    "Brake rotor measured 22.5mm, way below spec. Possible tool wear on Line 2.",
    "Wiring harness showing intermittent continuity failure, connector pins corroded.",
    "Paint surface has orange peel texture and fish-eye contamination on door panels.",
]

for query in test_queries:
    print(f"QUERY: {query}")
    result = rag_chain.invoke(query)
    print(f"ANSWER: {result}\n")
    print("-" * 80)


Record the retriever setup (endpoint, index, model, prompt) in an MLflow run so each setup run is traceable.


In [ ]:
# Record the retriever configuration in MLflow (no UC model registration)
with mlflow.start_run(run_name="service_bulletin_retriever"):
    mlflow.log_params({
        "vs_endpoint": VS_ENDPOINT_NAME,
        "vs_index": VS_INDEX_NAME,
        "llm_endpoint": LLM_MODEL,
        "top_k": 3,
    })
    mlflow.log_text(str(rag_prompt.messages), "rag_prompt.txt")
    print(f"✅ Retriever config logged to MLflow run {mlflow.active_run().info.run_id}")


## Sub-Agent 3: Quality SQL Agent (Text-to-SQL)

Converts natural language questions about quality data into SQL queries.


Table schemas passed into the text-to-SQL prompt. Every generated query must use fully qualified names.


In [ ]:
# Table schemas for context
table_schemas = f"""
Available tables in {catalog}.{schema}:

1. defect_history - 2,000 historical defect records
   Columns: defect_id, defect_date, plant_id, plant_name, line, shift, part_type, part_family,
            severity (Critical/Major/Minor), root_cause (Material/Process/Equipment/Human),
            status (Open/In Rework/Reworked/Scrapped/Released), batch_id, batch_size,
            affected_units, rework_cost_usd, resolution_days, inspector

2. plant_metrics - daily KPI rollups per plant (6 months, 5 plants)
   Columns: metric_date, plant_id, plant_name, ftq_rate, oee_rate, throughput_units,
            scrap_count, rework_count, planned_downtime_min, unplanned_downtime_min,
            total_downtime_min, shift_count

3. corrective_actions - 300 action records linked to inspections
   Columns: action_id, inspection_id, plant_id, part_type, action_type, action_description,
            assigned_to, action_date, completion_date, status, severity

4. quality_inspections - 500 inspection records
   Columns: inspection_id, inspection_date, plant_name, line, shift, inspector,
            part_type, measurement_value, measurement_unit, spec_min, spec_max, defect_notes
"""

sql_prompt = ChatPromptTemplate.from_template("""You are a SQL expert for manufacturing quality data on Databricks.
Convert the user's question into a Databricks SQL query.

{table_schemas}

Rules:
- Use fully qualified table names: """ + f"{catalog}.{schema}" + """.<table>
- Return ONLY a single SELECT or WITH query, no explanation
- No DDL or DML
- Use appropriate aggregations, filters, and ordering
- Limit results to 20 rows unless the user asks for more
- For date ranges, use DATE functions

QUESTION: {question}

SQL:""")

sql_chain = (
    {"question": RunnablePassthrough(), "table_schemas": lambda _: table_schemas}
    | sql_prompt
    | llm
    | StrOutputParser()
)


Generate SQL for sample questions and execute only if the guardrails pass.


In [ ]:
# Test SQL generation
sql_test_questions = [
    "What are the top 5 part types by total rework cost in the last 3 months?",
    "Show me the FTQ trend for Arlington Assembly over the last 30 days",
    "Which plant has the most critical defects and what's the primary root cause?",
    "How many open corrective actions are there by severity?",
]

for q in sql_test_questions:
    print(f"Q: {q}")
    sql = sql_chain.invoke(q)
    print(f"SQL: {sql}")
    try:
        safe_sql = sanitize_sql(sql, catalog, schema)
        result = spark.sql(safe_sql)
        display(result)
    except Exception as e:
        print(f"  SQL error: {e}")
    print("-" * 80)


Wrap SQL generation, guardrails, and execution in one helper, then run a sample question.


In [ ]:
# Wrap SQL agent as a function that executes and returns results
def sql_agent(question: str) -> str:
    """Convert question to SQL, execute, return results as JSON."""
    import json
    sql = sql_chain.invoke(question)
    try:
        safe_sql = sanitize_sql(sql, catalog, schema)
        df = spark.sql(safe_sql)
        rows = df.limit(20).toPandas().to_dict(orient="records")
        return json.dumps({
            "sql_query": safe_sql,
            "results": rows,
            "row_count": len(rows),
        }, default=str)
    except Exception as e:
        return json.dumps({
            "sql_query": sql,
            "error": str(e),
        })

# Test
import json
result = sql_agent("What's the average FTQ rate by plant?")
print(json.dumps(json.loads(result), indent=2))


## Summary


In [ ]:
print("✅ All 3 sub-agents built and tested:")
print("  1. Defect Classifier    — ai_query with constrained prompting")
print("  2. Bulletin Retriever   — Vector Search RAG with LangChain")
print("  3. Quality SQL Agent    — Text-to-SQL with Foundation Model")
print()
print("Next: Run 04_Supervisor_Agent.py — orchestration layer")
